# 07 - Explainable AI and Recovery Strategy Engine

## Explainability definition
SHAP quantifies how each feature contributes to a prediction for both global and local explanations.

## Business purpose
Collections teams need transparent reasons behind risk scores before acting.


## Definition
Explainability and strategy logic connect model outputs to actionable, auditable recovery decisions.

## Theory
This section explains the statistical or ML theory behind the technique and why it is useful in credit recovery operations.

## Mathematical Intuition
We translate the idea into score/probability/ranking logic so each metric can be interpreted quantitatively.

## Financial Intuition
We connect the method to borrower affordability, delinquency severity, collateral protection, and expected recoverable cashflows.

## Business Impact
We explain what this enables for collection managers, risk teams, and executives.

## Real-World Example
If risk is very high and collateral is weak, the strategy escalates from reminders to legal pathway preparation.

## Visual Explanation
Charts in this notebook show how model/segment behavior changes across borrower groups.

## Code Explanation
Every code cell below is paired with interpretation so beginners can connect implementation details to business outcomes.

## Interpretation of Results
We summarize what changed, why it matters, and how to act on it.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.loan_recovery import (
    DATA_PATH,
    FIGURES_DIR,
    MODELS_DIR,
    TABLES_DIR,
    REPORTS_DIR,
    LoanDataLoader,
    FeatureEngineer,
    LoanEDA,
    BorrowerSegmenter,
    ModelTrainer,
    ModelEvaluator,
    RecoveryStrategyEngine,
    ModelExplainer,
    DashboardBuilder,
    LazyPredictBenchmark,
    PyCaretWorkflow,
    FLAMLOptimizer,
    SmartLoanRecoveryPipeline,
    load_model,
)

sns.set_theme(style="whitegrid")


In [ ]:
def ensure_pipeline_artifacts() -> None:
    required = [
        TABLES_DIR / "manual_model_leaderboard.csv",
        TABLES_DIR / "feature_enriched_data.csv",
        MODELS_DIR / "best_manual_model.joblib",
    ]
    if not all(path.exists() for path in required):
        pipeline = SmartLoanRecoveryPipeline(data_path=DATA_PATH, random_state=42)
        pipeline.run()

ensure_pipeline_artifacts()


In [ ]:
enriched = pd.read_csv(TABLES_DIR / "feature_enriched_data.csv")
model = load_model(MODELS_DIR / "best_manual_model.joblib")

strategy = RecoveryStrategyEngine(model=model, feature_engineer=FeatureEngineer())
scored = strategy.score_portfolio(enriched)
assigned = strategy.assign_strategies(scored)
scenario_df = strategy.what_if_scenarios(enriched)

display(assigned[["Borrower_ID", "risk_score", "risk_tier", "recommended_strategy", "priority_rank"]].head(20))
display(scenario_df)


In [ ]:
split = FeatureEngineer().train_test_split(enriched, target_col="Recovery_Status", drop_cols=["Borrower_ID"])
explainer = ModelExplainer(model)
shap_out = explainer.explain(split.x_test)
shap_out


In [ ]:
from IPython.display import Image, HTML, display

for image_path in [
    FIGURES_DIR / "feature_importance.png",
    FIGURES_DIR / "shap_summary.png",
    FIGURES_DIR / "shap_waterfall.png",
]:
    if image_path.exists():
        display(Image(filename=str(image_path)))

force_html = FIGURES_DIR / "shap_force.html"
if force_html.exists():
    display(HTML(force_html.read_text()))


## Strategy logic
Risk tiers are not fixed constants. They are learned from portfolio quantiles, then mapped to operational actions.
